In [25]:
import os
import re
import pandas as pd

folder_raw = "../data/raw"
folder_processed = "../data/processed"
os.makedirs(folder_processed, exist_ok=True)

def buat_qa_pairs(no_perkara, pihak, pasal):
    """Fungsi pembantu Feature Engineering untuk membuat QA-Pairs sederhana"""
    qa_list = [
        f"Q: Apa nomor perkara kasus ini? | A: {no_perkara}.",
        f"Q: Siapa pihak yang bersengketa? | A: {pihak}.",
        f"Q: Apa pasal/rujukan hukum utamanya? | A: {pasal}."
    ]
    return " // ".join(qa_list)

def ekstrak_informasi(teks):
    teks_lc = teks.lower()

    # 1. Nomor Perkara
    pola_no = r'(?:nomor|no)[\s\:\.]*([0-9]+\s*[a-z\-]*\s*\/\s*[a-z\.\-]+\s*\/(?:[a-z]+\s*\/)?\s*[0-9]{4})'
    match_no = re.search(pola_no, teks_lc)
    no_perkara = re.sub(r'\s+', ' ', match_no.group(1)).upper().strip() if match_no else "TIDAK DITEMUKAN"

    # 2. Tanggal Putusan
    pola_tgl = r'tanggal\s+(\d{1,2}\s+[a-z]+\s+\d{4})'
    match_tgl = re.search(pola_tgl, teks_lc)
    tanggal = match_tgl.group(1).strip() if match_tgl else "TIDAK DITEMUKAN"

    # 3. PASAL & RUJUKAN HUKUM
    pasal_matches = re.findall(r'pasal\s+\d+(?:\s+ayat\s*\d+|\s+ayat\s*\(\d+\))?', teks_lc)
    if pasal_matches:
        pasal_unique = list(set([re.sub(r'\s+', ' ', p).strip().upper() for p in pasal_matches]))
        pasal = ", ".join(pasal_unique[:4])
    else:
        fallback_waris = re.findall(r'(\bkhi\b|\bkompilasi\s+hukum\b|\ban-nisa\b|\bal-quran\b|\bkuhperdata\b|\bbw\b)', teks_lc)
        if fallback_waris:
            pasal = "RUJUKAN: " + ", ".join(list(set([f.upper() for f in fallback_waris])))
        else:
            pasal = "HUKUM ADAT / FARAID"

    # 4. Pihak Sengketa Kasasi
    pemohon_match = re.search(r'(?:para\s+)?(?:pemohon\s+kasasi|para\s+pemohon\s+kasasi)\s*[:,\-]?\s*(.*?)\s+(?:melawan|selaku|dahulu|dalam|beralamat)', teks_lc)
    termohon_match = re.search(r'(?:termohon\s+kasasi|para\s+termohon\s+kasasi)\s*[:,\-]?\s*(.*?)\s+(?:dan|atau|membaca|menimbang|beralamat)', teks_lc)
    
    if pemohon_match and termohon_match:
        pemohon = re.sub(r'[^a-z\s\.,\-]', '', pemohon_match.group(1)).strip()[:35]
        termohon = re.sub(r'[^a-z\s\.,\-]', '', termohon_match.group(1)).strip()[:35]
        pihak = f"{pemohon.upper()} VS {termohon.upper()}"
    else:
        antara_match = re.search(r'antara\s*[:\.]?\s*([\w\s\.,\-]+)\s+melawan\s+([\w\s\.,\-]+)', teks_lc)
        if antara_match:
            pihak = f"{antara_match.group(1).strip()[:30].upper()} VS {antara_match.group(2).strip()[:30].upper()}"
        else:
            pihak = "TIDAK DITEMUKAN"

    # 5. Ringkasan Fakta (Konten Kunci 1)
    words = teks_lc.split()
    ringkasan_fakta = " ".join(words[:300]) + "..." if len(words) > 300 else teks_lc
    
    # 6. Argumen Hukum Utama / Amar Putusan (Konten Kunci 2)
    pola_amar = r'mengadili\s*(.*)'
    match_amar = re.search(pola_amar, teks_lc, re.DOTALL)
    # Diubah penamaannya agar pas dengan format "argumen_hukum_utama" di silabus
    argumen_hukum_utama = match_amar.group(1).strip()[:1000] + "..." if match_amar else "TIDAK DITEMUKAN"

    return no_perkara, tanggal, pasal, pihak, ringkasan_fakta, argumen_hukum_utama

print("✅ Cell 1: Fungsi Ekstraksi (Aman & Sesuai Silabus) Sukses Dimuat!")

✅ Cell 1: Fungsi Ekstraksi (Aman & Sesuai Silabus) Sukses Dimuat!


In [26]:
print("Memulai pembacaan dokumen dan feature engineering untuk Perdata Waris...")
data_kasus = []

file_list = [f for f in os.listdir(folder_raw) if f.endswith('.txt') and f.startswith('case_')]
file_list.sort()

for nama_file in file_list:
    path_txt = os.path.join(folder_raw, nama_file)
    with open(path_txt, 'r', encoding='utf-8') as f:
        teks_bersih = f.read()
    
    # Eksekusi fungsi ekstraksi dari cell 1
    no_perkara, tanggal, pasal, pihak, ringkasan_fakta, argumen_hukum_utama = ekstrak_informasi(teks_bersih)
    
    # --- iii. Feature Engineering ---
    # 1. Length (Jumlah kata total)
    jumlah_kata = len(teks_bersih.split())
    
    # 2. Bag-of-Words Sederhana (Menghitung frekuensi kata kunci krusial waris)
    frek_waris = teks_bersih.lower().count("waris")
    frek_ahli = teks_bersih.lower().count("ahli")
    frek_harta = teks_bersih.lower().count("harta")
    bag_of_words_mini = f"waris:{frek_waris}, ahli:{frek_ahli}, harta:{frek_harta}"
    
    # 3. QA-Pairs Sederhana
    qa_pairs = buat_qa_pairs(no_perkara, pihak, pasal)
    
    # Memasukkan ke list dengan nama kolom terstruktur sesuai tabel perintah
    data_kasus.append({
        'case_id': nama_file.replace('.txt', ''),
        'no_perkara': no_perkara,
        'tanggal': tanggal,
        'ringkasan_fakta': ringkasan_fakta,
        'argumen_hukum_utama': argumen_hukum_utama,
        'pasal': pasal,
        'pihak': pihak,
        'length_jumlah_kata': jumlah_kata,   
        'bag_of_words': bag_of_words_mini,
        'qa_pairs': qa_pairs,
        'text_full': teks_bersih
    })

# Mengubah hasil list menjadi DataFrame Pandas
df_cases = pd.DataFrame(data_kasus)
print(f"Selesai! Berhasil mengekstrak {len(df_cases)} kasus Perdata Waris.")

# Menampilkan kolom utama hasil ekstraksi untuk memastikan semua instruksi terpenuhi
display(df_cases[['case_id', 'no_perkara', 'tanggal', 'pasal', 'pihak', 'length_jumlah_kata', 'bag_of_words']].head())

Memulai pembacaan dokumen dan feature engineering untuk Perdata Waris...
Selesai! Berhasil mengekstrak 36 kasus Perdata Waris.


,case_id,no_perkara,tanggal,pasal,pihak,length_jumlah_kata,bag_of_words
0,case_001,6168 K/PDT/2025,12 agustus 2025,HUKUM ADAT / FARAID,DAHULU TERGUGAT I L A W A N . TIUR VS DAHULU ...,3950,"waris:46, ahli:22, harta:19"
1,case_002,6121 K/PDT/2025,16 agustus 2024,HUKUM ADAT / FARAID,PENGGUGAT L A W A N TJHAI KIM KWET VS TERGUGA...,4318,"waris:46, ahli:27, harta:2"
2,case_003,5651 K/PDT/2025,15 mei 2025,"PASAL 19 AYAT 1, PASAL 32 AYAT 2, PASAL 51, PA...",I VS DAHULU PARA PENGGUGAT D A N . KOPER,5148,"waris:29, ahli:15, harta:9"
3,case_004,5250 K/PDT/2025,11 mei 2021,HUKUM ADAT / FARAID,"L A W A N . LOUISA CLARA, BERTEMPAT VS D A N ....",2305,"waris:7, ahli:1, harta:2"
4,case_005,5165 K/PDT/2025,22 april 2025,HUKUM ADAT / FARAID,DAHULU PARA PENGGUGAT L A W A N IR. VS DAHULU ...,4462,"waris:17, ahli:10, harta:6"


In [27]:
path_csv = os.path.join(folder_processed, "cases.csv")
path_json = os.path.join(folder_processed, "cases.json")

# Simpan ke CSV
df_cases.to_csv(path_csv, index=False, encoding='utf-8')

# Simpan ke JSON dengan format rapi (berjarak/indented)
df_cases.to_json(path_json, orient='records', indent=4, force_ascii=False)

print("-" * 65)
print("🎉 TAHAP 2 SELESAI DENGAN SEMPURNAL!")
print(f" Dataset CSV sukses disimpan di  : {path_csv}")
print(f" Dataset JSON sukses disimpan di : {path_json}")
print("-" * 65)

-----------------------------------------------------------------
🎉 TAHAP 2 SELESAI DENGAN SEMPURNAL!
 Dataset CSV sukses disimpan di  : ../data/processed\cases.csv
 Dataset JSON sukses disimpan di : ../data/processed\cases.json
-----------------------------------------------------------------
